In [ ]:
# Cell 1 — clean install
import subprocess, os

subprocess.run(["pip", "install", "-q", "--upgrade",
                "transformers", "datasets", "accelerate", "torch"],
               check=True)

# Force restart kernel to clear all stale imports
os.kill(os.getpid(), 9)

In [1]:
# Cell 1 — fixed
#!pip install -q -U transformers datasets accelerate
# IMPORTANT: Runtime -> Restart session after this cell finishes

#from google.colab import drive
#drive.mount('/content/drive')

import os
BASE     = '/content/drive/MyDrive/SIB_BATTERYBERT'
CORPUS   = f'{BASE}/data/corpus_mlm.txt'
CKPT_OUT = f'{BASE}/checkpoints/sib-bert-mlm'
os.makedirs(CKPT_OUT, exist_ok=True)

with open(CORPUS) as f:
    lines = f.readlines()
print(f'Corpus sentences: {len(lines)}')

Corpus sentences: 12045


In [3]:
# Cell 2
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = 'allenai/scibert_scivocab_uncased'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

with open(CORPUS) as f:
    sentences = [l.strip() for l in f if l.strip()]

print(f'Tokenising {len(sentences)} sentences...')

def tokenize_fn(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=128,
        padding='max_length',
    )

raw_ds = Dataset.from_dict({'text': sentences})
tok_ds = raw_ds.map(tokenize_fn, batched=True,
                    remove_columns=['text'],
                    desc='Tokenising')

print(f'Dataset: {tok_ds}')

Tokenising 12045 sentences...


Tokenising:   0%|          | 0/12045 [00:00<?, ? examples/s]

Dataset: Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 12045
})


In [4]:
# Cell 3
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,   # standard BERT masking rate
)
# Quick sanity check
import torch
sample = tok_ds.select(range(4))
sample.set_format('torch')
batch = data_collator([sample[i] for i in range(4)])
print('input_ids shape :', batch['input_ids'].shape)
print('labels shape    :', batch['labels'].shape)
masked = (batch['labels'] != -100).sum().item()
print(f'Masked tokens in sample batch: {masked}')

input_ids shape : torch.Size([4, 128])
labels shape    : torch.Size([4, 128])
Masked tokens in sample batch: 11


In [5]:
# Cell 4
from transformers import (AutoModelForMaskedLM,
                          TrainingArguments, Trainer)

model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)
print(f'Model parameters: {model.num_parameters():,}')

# 90/10 train/eval split of corpus
split    = tok_ds.train_test_split(test_size=0.1, seed=42)
train_ds = split['train']
eval_ds  = split['test']
print(f'Train: {len(train_ds)}  Eval: {len(eval_ds)}')

args = TrainingArguments(
    output_dir                  = CKPT_OUT,
    num_train_epochs            = 3,
    per_device_train_batch_size = 64,
    per_device_eval_batch_size  = 64,
    learning_rate               = 2e-5,
    weight_decay                = 0.01,
    warmup_ratio                = 0.06,
    lr_scheduler_type           = 'cosine',
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    fp16                        = True,
    logging_steps               = 100,
    report_to                   = 'none',
    dataloader_num_workers      = 0,
)

trainer = Trainer(
    model            = model,
    args             = args,
    train_dataset    = train_ds,
    eval_dataset     = eval_ds,
    data_collator    = data_collator,
    processing_class = tokenizer,
)

print('Starting MLM pre-training...')
trainer.train()

pytorch_model.bin:   0%|          | 0.00/442M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture;

Model parameters: 133,859,300
Train: 10840  Eval: 1205


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=510, training_loss=1.7889626951778637, metrics={'train_runtime': 290.2864, 'train_samples_per_second': 112.027, 'train_steps_per_second': 1.757, 'total_flos': 2736985465774080.0, 'train_loss': 1.7889626951778637, 'epoch': 3.0})

In [6]:
# Cell 5
trainer.save_model(CKPT_OUT)
tokenizer.save_pretrained(CKPT_OUT)

print('Checkpoint saved to:', CKPT_OUT)
print('Files:')
for fname in sorted(os.listdir(CKPT_OUT)):
    size = os.path.getsize(f'{CKPT_OUT}/{fname}') / 1e6
    print(f'  {fname:<40} {size:.1f} MB')

# Reload + fill-mask sanity check
from transformers import AutoModelForMaskedLM, AutoTokenizer, pipeline
ckpt_model = AutoModelForMaskedLM.from_pretrained(CKPT_OUT)
ckpt_tok   = AutoTokenizer.from_pretrained(CKPT_OUT)
fill = pipeline('fill-mask', model=ckpt_model, tokenizer=ckpt_tok)
result = fill('Hard carbon is a promising [MASK] material for sodium-ion batteries.')
print('\nFill-mask sanity check:')
for r in result[:5]:
    print(f"  {r['score']:.3f}  {r['token_str']}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Checkpoint saved to: /content/drive/MyDrive/SIB_BATTERYBERT/checkpoints/sib-bert-mlm
Files:
  checkpoint-340                           0.0 MB
  checkpoint-510                           0.0 MB
  config.json                              0.0 MB
  model.safetensors                        535.5 MB
  tokenizer.json                           0.7 MB
  tokenizer_config.json                    0.0 MB
  training_args.bin                        0.0 MB


Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



Fill-mask sanity check:
  0.491  anode
  0.402  cathode
  0.093  electrode
  0.003  raw
  0.002  active
